# Create Retail Ontology

Create a Microsoft Fabric ontology item from core retail Silver tables in the attached Lakehouse.

## Included entity types
- Geography
- Store
- DistributionCenter
- Truck
- Customer
- Product
- Receipt
- OnlineOrder
- Promotion
- Payment
- CustomerSegment (gold / ML)
- ChurnPrediction (gold / ML)

## Included relationships
- Store -> Geography
- DistributionCenter -> Geography
- Customer -> Geography
- Truck -> DistributionCenter
- Receipt -> Customer
- Receipt -> Store
- Receipt -> Product (via receipt lines)
- OnlineOrder -> Customer
- OnlineOrder -> Product (via order lines)
- Promotion -> Receipt
- Payment -> Receipt
- CustomerSegment -> Customer
- ChurnPrediction -> Customer
- Store -> Product stockout risk (via gold stockout_risk)

ML-derived entities bind to the gold (`au`) schema; everything else binds to Silver.

By default the notebook deletes any ontology with the same name before recreating it.

In [ ]:
import base64
import json
import os
import random
import re
import time
import uuid

import sempy.fabric as fabric

In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env('LAKEHOUSE_NAME')
LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
def get_env_bool(var_name, default='true'):
    value = get_env(var_name, default)
    return str(value).strip().lower() in {'1', 'true', 'yes', 'y'}

def normalize_ontology_name(value, default='RetailOntology_AutoGen'):
    raw_value = str(value or default).strip()
    normalized = re.sub(r'[^A-Za-z0-9_]+', '_', raw_value)
    normalized = re.sub(r'_+', '_', normalized).strip('_')
    if not normalized:
        normalized = default
    if not normalized[0].isalpha():
        normalized = f'RetailOntology_{normalized}'
    normalized = normalized[:89].rstrip('_')
    if not normalized:
        normalized = default
    if raw_value != normalized:
        print(
            f"Normalized ONTOLOGY_NAME from {raw_value!r} to {normalized!r} "
            'to satisfy Fabric item naming rules.'
        )
    return normalized

SILVER_DB = get_env('SILVER_DB', 'ag')
GOLD_DB = get_env('GOLD_DB', 'au')
ONTOLOGY_NAME = normalize_ontology_name(get_env('ONTOLOGY_NAME', 'RetailOntology_AutoGen'))
ONTOLOGY_DESCRIPTION = get_env(
    'ONTOLOGY_DESCRIPTION',
    'Retail domain ontology generated from Lakehouse Silver tables.'
)
DELETE_EXISTING = get_env_bool('DELETE_EXISTING', 'true')
DELETE_WAIT_SECONDS = int(get_env('DELETE_WAIT_SECONDS', '30'))



print(f'SILVER_DB={SILVER_DB}')
print(f'GOLD_DB={GOLD_DB}')
print(f'LAKEHOUSE_NAME={LAKEHOUSE_NAME or "<auto-detect>"}')
print(f'ONTOLOGY_NAME={ONTOLOGY_NAME}')
print(f'DELETE_EXISTING={DELETE_EXISTING}')
print(f'DELETE_WAIT_SECONDS={DELETE_WAIT_SECONDS}')

In [ ]:
# =============================================================================
# RETAIL ENTITY / RELATIONSHIP MODEL
# =============================================================================

ENTITY_CONFIG = {
    'dim_geographies': {
        'entity_name': 'Geography',
        'key_candidates': ['geography_id', 'GeographyID', 'id', 'ID'],
        'display_candidates': ['city', 'City', 'region', 'Region', 'state', 'State', 'country', 'Country', 'id', 'ID'],
    },
    'dim_stores': {
        'entity_name': 'Store',
        'key_candidates': ['store_id', 'StoreID', 'id', 'ID'],
        'display_candidates': ['store_number', 'StoreNumber', 'store_name', 'StoreName', 'address', 'Address', 'id', 'ID'],
    },
    'dim_distribution_centers': {
        'entity_name': 'DistributionCenter',
        'key_candidates': ['dc_id', 'DCID', 'id', 'ID'],
        'display_candidates': ['dc_number', 'DCNumber', 'name', 'Name', 'address', 'Address', 'id', 'ID'],
    },
    'dim_trucks': {
        'entity_name': 'Truck',
        'key_candidates': ['truck_id', 'TruckID', 'id', 'ID'],
        'display_candidates': ['license_plate', 'LicensePlate', 'truck_number', 'TruckNumber', 'id', 'ID'],
    },
    'dim_customers': {
        'entity_name': 'Customer',
        'key_candidates': ['customer_id', 'CustomerID', 'id', 'ID'],
        'display_candidates': ['customer_name', 'CustomerName', 'loyalty_card', 'LoyaltyCard', 'email', 'Email', 'phone', 'Phone', 'id', 'ID'],
    },
    'dim_products': {
        'entity_name': 'Product',
        'key_candidates': ['product_id', 'ProductID', 'id', 'ID'],
        'display_candidates': ['product_name', 'ProductName', 'name', 'Name', 'brand', 'Brand', 'id', 'ID'],
    },
    'fact_receipts': {
        'entity_name': 'Receipt',
        'key_candidates': ['receipt_id_ext', 'ReceiptIdExt', 'receipt_id', 'ReceiptId', 'ReceiptID'],
        'display_candidates': ['receipt_id_ext', 'ReceiptIdExt', 'receipt_id', 'ReceiptId', 'ReceiptID'],
    },
    'fact_online_order_headers': {
        'entity_name': 'OnlineOrder',
        'key_candidates': ['order_id_ext', 'OrderIdExt', 'order_id', 'OrderId'],
        'display_candidates': ['order_id_ext', 'OrderIdExt', 'order_id', 'OrderId'],
    },
    # The generator emits one promotion_applied event per receipt, so
    # receipt_id_ext is unique here (trace_id is absent on historical rows).
    'fact_promotions': {
        'entity_name': 'Promotion',
        'key_candidates': ['receipt_id_ext', 'ReceiptId'],
        'display_candidates': ['promo_code', 'PromoCode'],
    },
    'fact_payments': {
        'entity_name': 'Payment',
        'key_candidates': ['transaction_id', 'payment_id', 'TransactionId'],
        'display_candidates': ['payment_method', 'PaymentMethod', 'transaction_id'],
    },
    # ML outputs live in the gold schema
    'customer_segments': {
        'entity_name': 'CustomerSegment',
        'schema_db': 'gold',
        'key_candidates': ['customer_id'],
        'display_candidates': ['segment_label', 'cluster_id', 'customer_id'],
    },
    'churn_predictions': {
        'entity_name': 'ChurnPrediction',
        'schema_db': 'gold',
        'key_candidates': ['customer_id'],
        'display_candidates': ['risk_category', 'customer_id'],
    },
}

RELATIONSHIPS = [
    {
        'name': 'StoreLocatedInGeography',
        'source_table': 'dim_stores',
        'source_entity': 'Store',
        'source_column_candidates': ['store_id', 'StoreID', 'id', 'ID'],
        'target_entity': 'Geography',
        'target_column_candidates': ['geography_id', 'GeographyID'],
    },
    {
        'name': 'DistributionCenterLocatedInGeography',
        'source_table': 'dim_distribution_centers',
        'source_entity': 'DistributionCenter',
        'source_column_candidates': ['dc_id', 'DCID', 'id', 'ID'],
        'target_entity': 'Geography',
        'target_column_candidates': ['geography_id', 'GeographyID'],
    },
    {
        'name': 'CustomerLocatedInGeography',
        'source_table': 'dim_customers',
        'source_entity': 'Customer',
        'source_column_candidates': ['customer_id', 'CustomerID', 'id', 'ID'],
        'target_entity': 'Geography',
        'target_column_candidates': ['geography_id', 'GeographyID'],
    },
    {
        'name': 'TruckAssignedToDistributionCenter',
        'source_table': 'dim_trucks',
        'source_entity': 'Truck',
        'source_column_candidates': ['truck_id', 'TruckID', 'id', 'ID'],
        'target_entity': 'DistributionCenter',
        'target_column_candidates': ['dc_id', 'DCID'],
    },
    {
        'name': 'ReceiptPlacedByCustomer',
        'source_table': 'fact_receipts',
        'source_entity': 'Receipt',
        'source_column_candidates': ['receipt_id_ext', 'ReceiptIdExt', 'receipt_id', 'ReceiptId', 'ReceiptID'],
        'target_entity': 'Customer',
        'target_column_candidates': ['customer_id', 'CustomerID'],
    },
    {
        'name': 'ReceiptAtStore',
        'source_table': 'fact_receipts',
        'source_entity': 'Receipt',
        'source_column_candidates': ['receipt_id_ext', 'ReceiptIdExt', 'receipt_id', 'ReceiptId', 'ReceiptID'],
        'target_entity': 'Store',
        'target_column_candidates': ['store_id', 'StoreID'],
    },
    {
        'name': 'ReceiptContainsProduct',
        'source_table': 'fact_receipt_lines',
        'source_entity': 'Receipt',
        'source_column_candidates': ['receipt_id_ext', 'ReceiptIdExt', 'receipt_id', 'ReceiptId', 'ReceiptID'],
        'target_entity': 'Product',
        'target_column_candidates': ['product_id', 'ProductID'],
    },
    {
        'name': 'OnlineOrderPlacedByCustomer',
        'source_table': 'fact_online_order_headers',
        'source_entity': 'OnlineOrder',
        'source_column_candidates': ['order_id_ext', 'OrderIdExt', 'order_id', 'OrderId'],
        'target_entity': 'Customer',
        'target_column_candidates': ['customer_id', 'CustomerID'],
    },
    {
        'name': 'OnlineOrderContainsProduct',
        'source_table': 'fact_online_order_lines',
        'source_entity': 'OnlineOrder',
        'source_column_candidates': ['order_id_ext', 'OrderIdExt', 'order_id', 'OrderId'],
        'target_entity': 'Product',
        'target_column_candidates': ['product_id', 'ProductID'],
    },
    {
        'name': 'PromotionAppliedToReceipt',
        'source_table': 'fact_promotions',
        'source_entity': 'Promotion',
        'source_column_candidates': ['receipt_id_ext', 'ReceiptId'],
        'target_entity': 'Receipt',
        'target_column_candidates': ['receipt_id_ext', 'ReceiptId'],
    },
    {
        'name': 'PaymentForReceipt',
        'source_table': 'fact_payments',
        'source_entity': 'Payment',
        'source_column_candidates': ['transaction_id', 'payment_id', 'TransactionId'],
        'target_entity': 'Receipt',
        'target_column_candidates': ['receipt_id_ext', 'ReceiptId'],
    },
    {
        'name': 'CustomerHasSegment',
        'source_table': 'customer_segments',
        'schema_db': 'gold',
        'source_entity': 'CustomerSegment',
        'source_column_candidates': ['customer_id'],
        'target_entity': 'Customer',
        'target_column_candidates': ['customer_id'],
    },
    {
        'name': 'CustomerHasChurnPrediction',
        'source_table': 'churn_predictions',
        'schema_db': 'gold',
        'source_entity': 'ChurnPrediction',
        'source_column_candidates': ['customer_id'],
        'target_entity': 'Customer',
        'target_column_candidates': ['customer_id'],
    },
    # Each gold stockout_risk row scores a (store, product) pair; relationship
    # properties are not supported by the ontology definition format, so the
    # risk attributes themselves stay in the gold table.
    {
        'name': 'StoreHasStockoutRiskForProduct',
        'source_table': 'stockout_risk',
        'schema_db': 'gold',
        'source_entity': 'Store',
        'source_column_candidates': ['store_id'],
        'target_entity': 'Product',
        'target_column_candidates': ['product_id'],
    },
]

EXCLUDED_SOURCE_COLUMNS = {'__index_level_0__'}

SPARK_TO_ONTOLOGY = {
    'string': 'String',
    'boolean': 'Boolean',
    'date': 'DateTime',
    'timestamp': 'DateTime',
    'double': 'Double',
    'float': 'Double',
    'byte': 'BigInt',
    'short': 'BigInt',
    'integer': 'BigInt',
    'int': 'BigInt',
    'long': 'BigInt',
    'bigint': 'BigInt',
    'smallint': 'BigInt',
    'tinyint': 'BigInt',
}

def resolve_schema_db(cfg):
    # Map a config 'schema_db' marker ('gold') to the actual schema name.
    return GOLD_DB if cfg.get('schema_db') == 'gold' else SILVER_DB

def build_table_name(table_name, schema=None):
    return f'{LAKEHOUSE_NAME}.{schema or SILVER_DB}.{table_name}'

def snake_case(name):
    value = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    value = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', value)
    value = re.sub(r'[^0-9a-zA-Z]+', '_', value).strip('_')
    value = re.sub(r'_+', '_', value)
    return value.lower()

def make_unique_name(base_name, used_names):
    candidate = base_name
    suffix = 2
    while candidate in used_names:
        candidate = f'{base_name}_{suffix}'
        suffix += 1
    used_names.add(candidate)
    return candidate

def resolve_schema_column(field_names, candidates, label):
    lookup = {field.lower(): field for field in field_names}
    for candidate in candidates:
        resolved = lookup.get(candidate.lower())
        if resolved:
            return resolved
    raise KeyError(
        f'{label}: none of {candidates} were found. Available columns: {field_names}'
    )

def map_spark_type(data_type):
    simple_type = data_type.simpleString()
    if simple_type.startswith('decimal'):
        return 'Double'
    if simple_type.startswith(('array', 'map', 'struct', 'binary')):
        return None
    return SPARK_TO_ONTOLOGY.get(simple_type)

_used_ids = set()
random.seed(42)

def generate_id():
    while True:
        id_val = random.randint(10**12, 10**15)
        if id_val not in _used_ids:
            _used_ids.add(id_val)
            return str(id_val)

def to_base64(document):
    if isinstance(document, (dict, list)):
        payload = json.dumps(document).encode('utf-8')
    else:
        payload = str(document).encode('utf-8')
    return base64.b64encode(payload).decode('utf-8')

In [ ]:
# =============================================================================
# FABRIC CONTEXT
# =============================================================================

client = fabric.FabricRestClient()
WORKSPACE_ID = fabric.get_notebook_workspace_id()

WORKSPACE_NAME = WORKSPACE_ID

lakehouse_response = client.get(f'v1/workspaces/{WORKSPACE_ID}/lakehouses')
lakehouse_response.raise_for_status()
lakehouses = lakehouse_response.json().get('value', [])

if not lakehouses:
    raise ValueError(f'No lakehouses were found in workspace {WORKSPACE_NAME}')

if LAKEHOUSE_NAME:
    matching_lakehouses = [
        lakehouse
        for lakehouse in lakehouses
        if lakehouse.get('displayName', '').lower() == LAKEHOUSE_NAME.lower()
    ]
    if not matching_lakehouses:
        available = ', '.join(sorted(lakehouse.get('displayName', '<unnamed>') for lakehouse in lakehouses))
        raise ValueError(
            f'LAKEHOUSE_NAME={LAKEHOUSE_NAME} was not found in workspace {WORKSPACE_NAME}. '
            f'Available lakehouses: {available}'
        )
    lakehouse = matching_lakehouses[0]
elif len(lakehouses) == 1:
    lakehouse = lakehouses[0]
else:
    available = ', '.join(sorted(lakehouse.get('displayName', '<unnamed>') for lakehouse in lakehouses))
    raise ValueError(
        'LAKEHOUSE_NAME was not provided and multiple lakehouses were found. '
        f'Set LAKEHOUSE_NAME to one of: {available}'
    )

LAKEHOUSE_ID = lakehouse['id']
LAKEHOUSE_NAME = lakehouse['displayName']

print(f'Workspace: {WORKSPACE_NAME} ({WORKSPACE_ID})')
print(f'Lakehouse: {LAKEHOUSE_NAME} ({LAKEHOUSE_ID})')

In [ ]:
# =============================================================================
# READ SCHEMAS AND BUILD ONTOLOGY DEFINITION
# =============================================================================

table_field_names = {}
entity_type_ids = {}
property_ids = {}
key_property_ids = {}
entity_metadata = {}
relationship_metadata = []
parts = [
    {
        'path': '.platform',
        'payload': to_base64({
            'metadata': {
                'type': 'Ontology',
                'displayName': ONTOLOGY_NAME,
            }
        }),
        'payloadType': 'InlineBase64',
    },
    {'path': 'definition.json', 'payload': to_base64({}), 'payloadType': 'InlineBase64'},
]

print('Resolving entity schemas...')
for table_name, cfg in ENTITY_CONFIG.items():
    schema_db = resolve_schema_db(cfg)
    full_table_name = build_table_name(table_name, schema_db)
    schema_fields = spark.table(full_table_name).schema.fields
    field_names = [field.name for field in schema_fields]
    table_field_names[full_table_name] = field_names

    key_column = resolve_schema_column(field_names, cfg['key_candidates'], f'{full_table_name} key column')

    try:
        display_column = resolve_schema_column(
            field_names,
            cfg['display_candidates'],
            f'{full_table_name} display column'
        )
    except KeyError:
        display_column = key_column

    used_property_names = set()
    compatible_columns = []
    skipped_columns = []
    for field in schema_fields:
        if field.name in EXCLUDED_SOURCE_COLUMNS:
            skipped_columns.append(field.name)
            continue
        value_type = map_spark_type(field.dataType)
        if value_type is None:
            skipped_columns.append(field.name)
            continue
        compatible_columns.append({
            'source_name': field.name,
            'name': make_unique_name(snake_case(field.name), used_property_names),
            'valueType': value_type,
        })

    compatible_source_names = {column['source_name'] for column in compatible_columns}
    if key_column not in compatible_source_names:
        raise ValueError(f'{full_table_name} key column {key_column} has an unsupported ontology type')
    if display_column not in compatible_source_names:
        display_column = key_column

    entity_metadata[cfg['entity_name']] = {
        'table_name': table_name,
        'schema_db': schema_db,
        'full_table_name': full_table_name,
        'key_column': key_column,
        'display_column': display_column,
        'columns': compatible_columns,
    }

    print(
        f"  - {cfg['entity_name']}: table={full_table_name}, "
        f"properties={len(compatible_columns)}, key={key_column}, display={display_column}, "
        f"skipped={len(skipped_columns)}"
    )

print()
print('Resolving relationship bindings...')
for rel in RELATIONSHIPS:
    source_table = rel['source_table']
    rel_schema_db = resolve_schema_db(rel)
    rel_full_table_name = build_table_name(source_table, rel_schema_db)
    if rel_full_table_name not in table_field_names:
        source_schema_fields = spark.table(rel_full_table_name).schema.fields
        table_field_names[rel_full_table_name] = [field.name for field in source_schema_fields]

    source_field_names = table_field_names[rel_full_table_name]
    resolved_source_column = resolve_schema_column(
        source_field_names,
        rel['source_column_candidates'],
        f"{rel_full_table_name} source column for {rel['name']}"
    )
    resolved_target_column = resolve_schema_column(
        source_field_names,
        rel['target_column_candidates'],
        f"{rel_full_table_name} target column for {rel['name']}"
    )

    relationship_metadata.append({
        'name': rel['name'],
        'source_table': source_table,
        'schema_db': rel_schema_db,
        'full_table_name': rel_full_table_name,
        'source_entity': rel['source_entity'],
        'target_entity': rel['target_entity'],
        'source_column': resolved_source_column,
        'target_column': resolved_target_column,
    })

    print(
        f"  - {rel['name']}: {rel['source_entity']}->{rel['target_entity']} "
        f"via {rel_full_table_name}({resolved_source_column}, {resolved_target_column})"
    )

print()
print('Building ontology definition parts...')
for entity_name, metadata in entity_metadata.items():
    entity_type_id = generate_id()
    entity_type_ids[entity_name] = entity_type_id

    properties = []
    for column in metadata['columns']:
        prop_id = generate_id()
        property_ids[(entity_name, column['source_name'])] = prop_id
        properties.append({
            'id': prop_id,
            'name': column['name'],
            'redefines': None,
            'baseTypeNamespaceType': None,
            'valueType': column['valueType'],
        })

    key_prop_id = property_ids[(entity_name, metadata['key_column'])]
    key_property_ids[entity_name] = key_prop_id
    display_prop_id = property_ids[(entity_name, metadata['display_column'])]

    entity_def = {
        'id': entity_type_id,
        'namespace': 'usertypes',
        'baseEntityTypeId': None,
        'name': entity_name,
        'entityIdParts': [key_prop_id],
        'displayNamePropertyId': display_prop_id,
        'namespaceType': 'Custom',
        'visibility': 'Visible',
        'properties': properties,
        'timeseriesProperties': [],
    }
    parts.append({
        'path': f'EntityTypes/{entity_type_id}/definition.json',
        'payload': to_base64(entity_def),
        'payloadType': 'InlineBase64',
    })

    binding_id = str(uuid.uuid4())
    data_binding = {
        'id': binding_id,
        'dataBindingConfiguration': {
            'dataBindingType': 'NonTimeSeries',
            'propertyBindings': [
                {
                    'sourceColumnName': column['source_name'],
                    'targetPropertyId': property_ids[(entity_name, column['source_name'])],
                }
                for column in metadata['columns']
            ],
            'sourceTableProperties': {
                'sourceType': 'LakehouseTable',
                'workspaceId': WORKSPACE_ID,
                'itemId': LAKEHOUSE_ID,
                'sourceTableName': metadata['table_name'],
                'sourceSchema': metadata['schema_db'],
            },
        },
    }
    parts.append({
        'path': f'EntityTypes/{entity_type_id}/DataBindings/{binding_id}.json',
        'payload': to_base64(data_binding),
        'payloadType': 'InlineBase64',
    })
    print(f"  - Entity {entity_name}: {len(properties)} properties")

for rel in relationship_metadata:
    rel_id = generate_id()
    rel_def = {
        'namespace': 'usertypes',
        'id': rel_id,
        'name': rel['name'],
        'namespaceType': 'Custom',
        'source': {'entityTypeId': entity_type_ids[rel['source_entity']]},
        'target': {'entityTypeId': entity_type_ids[rel['target_entity']]},
    }
    parts.append({
        'path': f'RelationshipTypes/{rel_id}/definition.json',
        'payload': to_base64(rel_def),
        'payloadType': 'InlineBase64',
    })

    ctx_id = str(uuid.uuid4())
    contextualization = {
        'id': ctx_id,
        'dataBindingTable': {
            'workspaceId': WORKSPACE_ID,
            'itemId': LAKEHOUSE_ID,
            'sourceTableName': rel['source_table'],
            'sourceSchema': rel['schema_db'],
            'sourceType': 'LakehouseTable',
        },
        'sourceKeyRefBindings': [
            {
                'sourceColumnName': rel['source_column'],
                'targetPropertyId': key_property_ids[rel['source_entity']],
            }
        ],
        'targetKeyRefBindings': [
            {
                'sourceColumnName': rel['target_column'],
                'targetPropertyId': key_property_ids[rel['target_entity']],
            }
        ],
    }
    parts.append({
        'path': f'RelationshipTypes/{rel_id}/Contextualizations/{ctx_id}.json',
        'payload': to_base64(contextualization),
        'payloadType': 'InlineBase64',
    })
    print(f"  - Relationship {rel['name']}: {rel['source_entity']}->{rel['target_entity']}")

print()
print(f'Prepared {len(entity_metadata)} entity types, {len(relationship_metadata)} relationships, {len(parts)} total parts.')

In [ ]:
# =============================================================================
# CREATE OR RECREATE THE ONTOLOGY ITEM
# =============================================================================

resp = client.get(f'v1/workspaces/{WORKSPACE_ID}/ontologies')
resp.raise_for_status()
existing = next(
    (ontology for ontology in resp.json().get('value', []) if ontology.get('displayName') == ONTOLOGY_NAME),
    None,
)

if existing and not DELETE_EXISTING:
    raise ValueError(
        f"Ontology '{ONTOLOGY_NAME}' already exists. Set DELETE_EXISTING=true to replace it."
    )

if existing:
    print(f"Deleting existing ontology '{ONTOLOGY_NAME}' (id={existing['id']})...")
    delete_response = client.delete(f"v1/workspaces/{WORKSPACE_ID}/ontologies/{existing['id']}")
    delete_response.raise_for_status()
    print(f'Waiting {DELETE_WAIT_SECONDS} seconds for Fabric cleanup...')
    time.sleep(DELETE_WAIT_SECONDS)

payload = {
    'displayName': ONTOLOGY_NAME,
    'description': ONTOLOGY_DESCRIPTION,
    'definition': {'parts': parts},
}

print(f"Creating ontology '{ONTOLOGY_NAME}'...")
create_response = client.post(f'v1/workspaces/{WORKSPACE_ID}/ontologies', json=payload)

if create_response.status_code == 201:
    result = create_response.json()
    print(f"Ontology created. ID: {result.get('id', 'N/A')}")
elif create_response.status_code == 202:
    operation_id = create_response.headers.get('x-ms-operation-id')
    retry_after = int(create_response.headers.get('Retry-After', '30'))
    print(f'Operation {operation_id} accepted. Polling every {retry_after} seconds...')

    while True:
        time.sleep(retry_after)
        poll_response = client.get(f'v1/operations/{operation_id}')
        poll_response.raise_for_status()
        status = poll_response.json().get('status')
        print(f'  Status: {status}')

        if status == 'Succeeded':
            result_response = client.get(f'v1/operations/{operation_id}/result')
            if result_response.status_code == 200:
                print(f"Ontology created. ID: {result_response.json().get('id', 'N/A')}")
            else:
                print('Ontology creation succeeded.')
            break
        if status in ('Failed', 'Cancelled'):
            failure_details = poll_response.json()
            raise RuntimeError(f'Ontology creation {status.lower()}: {failure_details}')
else:
    print(f'HTTP {create_response.status_code}: {create_response.text}')
    create_response.raise_for_status()